# Batch No-GUI Vascular Pipeline Test Notebook

This notebook is self-contained: all pipeline code is defined in Cell 2, then configured and executed in later cells.

Recommended kernel/environment: **clean_vascumap**.

In [16]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import json
import math

import numpy as np
import pandas as pd
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
from liffile import LifFile
from monai.inferers import SlidingWindowInferer, SliceInferer
from monai.networks.nets import UNet
from scipy.ndimage import zoom as ndi_zoom
from scipy.ndimage import rotate, binary_dilation
from skimage import util
from skimage.draw import line
from skimage.filters import threshold_triangle, median, sobel, gaussian, threshold_yen, apply_hysteresis_threshold
from skimage.measure import label, regionprops_table, regionprops, moments_central
from skimage.morphology import disk, remove_small_objects, remove_small_holes, closing
from skimage.transform import ProjectiveTransform, warp, probabilistic_hough_line
import segmentation_models_pytorch as smp

try:
    import cupy as cp
    from cupyx.scipy import ndimage as cpx_ndimage
    HAS_CUPY = True
except Exception:
    cp = None
    cpx_ndimage = None
    HAS_CUPY = False


@dataclass
class InputImage:
    source_path: Path
    source_kind: str
    image_index: Optional[int]
    image_name: str


@dataclass
class BatchConfig:
    input_dir: Path
    output_dir: Path
    pix2pix_ckpt: Path
    seg_ckpt: Path
    device: str = "cuda"
    device_width_um: float = 40.0
    output_voxel_um: tuple[float, float, float] = (2.0, 2.0, 2.0)
    pix2pix_voxel_um: tuple[float, float, float] = (5.0, 2.0, 2.0)
    unet_input_voxel_um: tuple[float, float, float] = (2.0, 2.0, 2.0)
    focus_n_sampling: int = 10
    focus_patch: int = 50
    mask_sigma: float = 2.0
    mask_frac_thresh: float = 0.70
    low_frac: float = 0.82
    high_frac: float = 1.10
    smooth_window: int = 5
    bin_size: float = 2.0
    min_run_frac: float = 0.25
    typical_pct: float = 50.0
    line_length: int = 400
    line_gap: int = 900
    hough_threshold: int = 70
    use_gpu_resize: bool = True
    mask_central_region: bool = False
    overlay_max_dim: int = 1024


def _step(axis, xa):
    if axis not in xa.coords or xa.coords[axis].size < 2:
        return None
    return float(xa.coords[axis][1] - xa.coords[axis][0])


def read_voxel_size_um(item: InputImage) -> tuple[Optional[float], Optional[float], Optional[float]]:
    z_um = y_um = x_um = None
    if item.source_kind == "lif":
        try:
            with LifFile(item.source_path) as lif:
                idx = int(item.image_index if item.image_index is not None else 0)
                if idx < 0 or idx >= len(lif.images):
                    return (None, None, None)
                img = lif.images[idx]
                xa = img.asxarray()
                x_step = _step("X", xa)
                y_step = _step("Y", xa)
                z_step = _step("Z", xa)
                x_um = x_step * 1e6 if x_step is not None else None
                y_um = y_step * 1e6 if y_step is not None else None
                z_um = z_step * 1e6 if z_step is not None else None
        except Exception:
            return (None, None, None)
        return (z_um, y_um, x_um)

    try:
        with tifffile.TiffFile(str(item.source_path)) as tif:
            tif_tags = {tag.name: tag.value for tag in tif.pages[0].tags.values()}
            if "XResolution" in tif_tags:
                xres = tif_tags["XResolution"]
                x_um = 1.0 / (float(xres[0]) / float(xres[1]))
            if "YResolution" in tif_tags:
                yres = tif_tags["YResolution"]
                y_um = 1.0 / (float(yres[0]) / float(yres[1]))
            try:
                z_um = float(str(tif_tags["IJMetadata"]).split("nscales=")[1].split(",")[2].split("\\nunit")[0])
            except Exception:
                try:
                    z_um = float(str(tif_tags["ImageDescription"]).split("spacing=")[1].split("loop")[0])
                except Exception:
                    z_um = None
    except Exception:
        return (None, None, None)

    if x_um is None and y_um is not None:
        x_um = y_um
    if y_um is None and x_um is not None:
        y_um = x_um
    return (z_um, y_um, x_um)


def has_valid_voxel_um(voxel) -> bool:
    for v in voxel:
        if v is None:
            return False
        try:
            vf = float(v)
        except Exception:
            return False
        if not np.isfinite(vf) or vf <= 0:
            return False
    return True


def iterate_input_images(input_dir: Path) -> list[InputImage]:
    images = []
    for p in sorted(input_dir.rglob("*")):
        if not p.is_file():
            continue
        suf = p.suffix.lower()
        if suf in (".tif", ".tiff"):
            images.append(InputImage(p, "tif", None, p.name))
        elif suf == ".lif":
            try:
                with LifFile(p) as lif:
                    for idx, img in enumerate(lif.images):
                        name = img.name if getattr(img, "name", None) else f"image_{idx}"
                        images.append(InputImage(p, "lif", idx, name))
            except Exception:
                continue
    return images


def load_stack(item: InputImage) -> np.ndarray:
    if item.source_kind == "lif":
        with LifFile(item.source_path) as lif:
            arr = np.asarray(lif.images[int(item.image_index)].asarray())
        if arr.ndim == 2:
            arr = arr[np.newaxis, ...]
        elif arr.ndim == 3:
            arr = arr if arr.shape[0] < 64 else arr[np.newaxis, ...]
        elif arr.ndim != 4:
            raise RuntimeError(f"Unsupported LIF array shape: {arr.shape}")
    else:
        arr = np.asarray(tifffile.imread(str(item.source_path)))

    if item.source_kind != "lif":
        if arr.ndim == 2:
            arr = arr[np.newaxis, ...]
        elif arr.ndim == 4 and arr.shape[-1] in (3, 4):
            arr = np.mean(arr, axis=-1)
        elif arr.ndim == 3 and arr.shape[-1] in (3, 4) and arr.shape[0] > 32 and arr.shape[1] > 32:
            arr = np.mean(arr, axis=-1)[np.newaxis, ...]

    if arr.ndim != 3:
        raise RuntimeError(f"Unsupported image shape: {arr.shape}")
    return np.asarray(arr, dtype=np.float32)


def to_gray(im: np.ndarray) -> np.ndarray:
    return im if im.ndim == 2 else np.mean(im, axis=-1)


def focus_score(patch: np.ndarray) -> float:
    return float(np.std(sobel(np.asarray(to_gray(patch), dtype=float))))


def curved_plane_refocus(stack_zyx: np.ndarray, grid: int = 20, patch: int = 50):
    Z, H, W = stack_zyx.shape
    ys = np.linspace(patch // 2, H - patch // 2 - 1, grid).astype(int)
    xs = np.linspace(patch // 2, W - patch // 2 - 1, grid).astype(int)
    pts, zs = [], []
    for y in ys:
        for x in xs:
            sl = (slice(y - patch // 2, y + patch // 2), slice(x - patch // 2, x + patch // 2))
            f = np.array([focus_score(stack_zyx[z][sl]) for z in range(Z)], dtype=np.float32)
            pts.append((x, y))
            zs.append(int(np.argmax(f)))

    if len(zs) < 6:
        scores = [np.std(sobel(to_gray(stack_zyx[z]))) for z in range(Z)]
        z0 = int(np.argmax(scores))
        return to_gray(stack_zyx[z0]), np.full((H, W), z0, dtype=np.int16), np.array(pts), np.array(zs)

    pts = np.array(pts, dtype=np.float32)
    zs = np.array(zs, dtype=np.float32)
    Xc = pts[:, 0] - pts[:, 0].mean()
    Yc = pts[:, 1] - pts[:, 1].mean()
    s = max(W, H)
    Xn = Xc / s
    Yn = Yc / s
    B = np.column_stack((Xn**2, Yn**2, Xn * Yn, Xn, Yn, np.ones_like(Xn)))
    coeffs, *_ = np.linalg.lstsq(B, zs, rcond=None)
    Xg, Yg = np.meshgrid(np.arange(W), np.arange(H))
    mx = pts[:, 0].mean()
    my = pts[:, 1].mean()
    Xg_n = (Xg - mx) / s
    Yg_n = (Yg - my) / s
    zmap = coeffs[0] * Xg_n**2 + coeffs[1] * Yg_n**2 + coeffs[2] * Xg_n * Yg_n + coeffs[3] * Xg_n + coeffs[4] * Yg_n + coeffs[5]
    zmap = np.clip(np.rint(zmap).astype(np.int16), 0, Z - 1)
    img = np.take_along_axis(stack_zyx, zmap[None, :, :], axis=0)[0]
    return img, zmap, pts, zs


def _order_corners_clockwise(corners_xy: np.ndarray):
    c = np.asarray(corners_xy, dtype=float)
    centroid = c.mean(axis=0)
    angles = np.arctan2(c[:, 1] - centroid[1], c[:, 0] - centroid[0])
    c = c[np.argsort(angles)]
    start = np.argmin(c[:, 0] + c[:, 1])
    return np.roll(c, -start, axis=0)


def _signed_orientation(region):
    img = region.image.astype(float)
    mu = moments_central(img)
    angle_rad = 0.5 * np.arctan2(2 * mu[1, 1], mu[2, 0] - mu[0, 2])
    return np.rad2deg(angle_rad)


def _mask_out_organoid(in_focus_plane, mask_sigma: float, mask_frac_thresh: float):
    inverted = gaussian(util.invert(np.asarray(in_focus_plane, dtype=np.float32)), sigma=float(mask_sigma), mode="nearest", preserve_range=True).astype(np.float32)
    H, W = in_focus_plane.shape[:2]
    xy_area = H * W
    cyi, cxi = H // 2, W // 2
    r = int(min(H, W) * 0.1)
    yy, xx = np.ogrid[:H, :W]
    central_roi = (yy - cyi) ** 2 + (xx - cxi) ** 2 <= r**2
    thresh = threshold_yen(inverted)
    labelled = label(inverted > thresh)
    props = regionprops(labelled)
    if len(props) == 0:
        return np.zeros((H, W), dtype=bool)

    def score(p):
        overlap = np.sum(labelled[central_roi] == p.label)
        py, px = p.centroid
        dist2 = (py - cyi) ** 2 + (px - cxi) ** 2
        return (-overlap, dist2)

    best_prop = min(props, key=score)
    if (best_prop.area > xy_area * mask_frac_thresh) or (best_prop.solidity < 0.5):
        thresh = threshold_triangle(inverted)
        labelled = label(inverted > thresh)
        props = regionprops(labelled)
        if len(props) == 0:
            return np.zeros((H, W), dtype=bool)
        best_prop = min(props, key=score)

    organoid_region = labelled == best_prop.label
    organoid_region = remove_small_holes(organoid_region, area_threshold=50000)
    return closing(organoid_region, disk(5))


def _oriented_rect_corners(mask: np.ndarray, cfg: BatchConfig):
    ys, xs = np.nonzero(mask)
    if xs.size == 0:
        return None
    cx, cy = xs.mean(), ys.mean()
    mu = moments_central(mask.astype(np.uint8))
    angle_rad = 0.5 * np.arctan2(2 * mu[1, 1], (mu[0, 2] - mu[2, 0]))
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    dx = xs - cx
    dy = ys - cy
    u = c * dx + s * dy
    v = -s * dx + c * dy
    u_span = float(u.max() - u.min())
    v_span = float(v.max() - v.min())
    if v_span >= u_span:
        long, short, long_name = v, u, "v"
    else:
        long, short, long_name = u, v, "u"

    short_min_full = float(short.min())
    short_max_full = float(short.max())
    long_bin = np.floor(long / cfg.bin_size).astype(int)
    bins, inv = np.unique(long_bin, return_inverse=True)
    short_min = np.full(bins.shape, np.inf)
    short_max = np.full(bins.shape, -np.inf)
    np.minimum.at(short_min, inv, short)
    np.maximum.at(short_max, inv, short)
    widths = short_max - short_min

    if cfg.smooth_window > 1:
        w = int(cfg.smooth_window)
        if w % 2 == 0:
            w += 1
        pad = w // 2
        widths_s = np.convolve(np.pad(widths, (pad, pad), mode="edge"), np.ones(w) / w, mode="valid")
    else:
        widths_s = widths

    typical_width = float(np.percentile(widths_s, cfg.typical_pct))
    low_thr = cfg.low_frac * typical_width
    high_thr = cfg.high_frac * typical_width
    keep = (~(widths_s < low_thr)) & (~(widths_s > high_thr))
    too_wide = widths_s > high_thr

    best_start = best_end = None
    best_len = 0
    cur_start = cur_end = None
    for i in range(len(bins)):
        if not keep[i]:
            if cur_start is not None:
                L = cur_end - cur_start + 1
                if L > best_len:
                    best_len = L
                    best_start, best_end = cur_start, cur_end
                cur_start = cur_end = None
            continue
        if cur_start is None:
            cur_start = cur_end = bins[i]
        elif bins[i] == cur_end + 1:
            cur_end = bins[i]
        else:
            L = cur_end - cur_start + 1
            if L > best_len:
                best_len = L
                best_start, best_end = cur_start, cur_end
            cur_start = cur_end = bins[i]

    if cur_start is not None:
        L = cur_end - cur_start + 1
        if L > best_len:
            best_len = L
            best_start, best_end = cur_start, cur_end

    total_len_bins = int(bins.max() - bins.min() + 1)
    min_run_bins = int(np.ceil(cfg.min_run_frac * total_len_bins))

    if best_start is None or best_len < min_run_bins:
        long_min = float(long.min())
        long_max = float(long.max())
        crop_width = False
    else:
        long_min = float(best_start * cfg.bin_size)
        long_max = float((best_end + 1) * cfg.bin_size)
        band_mask_bins = (bins >= best_start) & (bins <= best_end)
        crop_width = bool(np.any(too_wide & (~band_mask_bins)))

    in_long_band = (long >= long_min) & (long <= long_max)
    if crop_width:
        short_min_use = float(short[in_long_band].min())
        short_max_use = float(short[in_long_band].max())
    else:
        short_min_use = short_min_full
        short_max_use = short_max_full

    if long_name == "v":
        umin, umax = short_min_use, short_max_use
        vmin, vmax = long_min, long_max
    else:
        umin, umax = long_min, long_max
        vmin, vmax = short_min_use, short_max_use

    corners_uv = np.array([[umin, vmin], [umax, vmin], [umax, vmax], [umin, vmax]], dtype=float)
    x = cx + c * corners_uv[:, 0] - s * corners_uv[:, 1]
    y = cy + s * corners_uv[:, 0] + c * corners_uv[:, 1]
    return np.stack([x, y], axis=1)


def _corners_touch_border(corners_xy: np.ndarray, shape, margin=0):
    H, W = shape
    x = corners_xy[:, 0]
    y = corners_xy[:, 1]
    return (x <= margin).any() or (x >= (W - 1 - margin)).any() or (y <= margin).any() or (y >= (H - 1 - margin)).any()


def segment_from_plane_conventional(in_focus_plane: np.ndarray, cfg: BatchConfig, mask_central_region: bool, return_debug: bool = False):
    flag = False
    plane = to_gray(in_focus_plane)

    median_thresholded = median(np.asarray(plane, dtype=np.float32), footprint=disk(7)).astype(np.float32)
    sobel_operated = sobel(median_thresholded).astype(np.float32)
    thresh = threshold_triangle(sobel_operated)
    binary = sobel_operated > thresh

    h, w = plane.shape[:2]
    binary[h // 3 : 2 * (h // 3), w // 3 : 2 * (w // 3)] = 0

    organoid_region = None
    if bool(mask_central_region):
        organoid_region = _mask_out_organoid(plane, cfg.mask_sigma, cfg.mask_frac_thresh)
        binary[organoid_region] = 0

    labels = label(binary)
    data = regionprops_table(labels, binary, properties=("label", "area", "eccentricity"))
    condition = (data["area"] > 100) & (data["eccentricity"] > 0.5)
    labels_to_dilate = util.map_array(labels, data["label"], data["label"] * condition)

    dilated_output = np.zeros_like(labels, dtype=np.uint8)
    base_selem = np.zeros((31, 31), dtype=bool)
    base_selem[15, :] = 1
    pad = base_selem.shape[0] // 2

    for region in regionprops(labels_to_dilate):
        angle_to_rotate = _signed_orientation(region)
        rotated_selem = rotate(base_selem.astype(float), angle=90 + angle_to_rotate, reshape=False, order=0) > 0.5

        minr, minc, maxr, maxc = region.bbox
        r0 = max(minr - pad, 0)
        r1 = min(maxr + pad, labels_to_dilate.shape[0])
        c0 = max(minc - pad, 0)
        c1 = min(maxc + pad, labels_to_dilate.shape[1])

        mask_roi = labels_to_dilate[r0:r1, c0:c1] == region.label
        dilated = binary_dilation(mask_roi.astype(bool), structure=rotated_selem.astype(bool))
        dilated_output[r0:r1, c0:c1][dilated] = 255

    post_dilation_mask = np.logical_or(dilated_output, binary)
    clean_labels = label(~post_dilation_mask)
    props = regionprops(clean_labels)
    largest_prop = max(props, key=lambda p: p.area)
    device_mask = clean_labels == largest_prop.label

    edges = reconstructed = reconstructed_mask = None
    new_corners = None

    corners = _oriented_rect_corners(device_mask, cfg)
    if corners is None or _corners_touch_border(corners, device_mask.shape, margin=5):
        flag = True
        edges = remove_small_objects(labels_to_dilate > 0)
        segs = probabilistic_hough_line(
            edges,
            line_length=cfg.line_length,
            line_gap=cfg.line_gap,
            threshold=cfg.hough_threshold,
        )
        reconstructed = np.zeros_like(edges, dtype=bool)
        for (x0, y0), (x1, y1) in segs:
            rr, cc = line(y0, x0, y1, x1)
            reconstructed[rr, cc] = True

        reconstructed_mask = np.logical_or(reconstructed, post_dilation_mask)
        updated_clean_labels = label(~reconstructed_mask)
        props = regionprops(updated_clean_labels)
        largest_prop = max(props, key=lambda p: p.area)
        new_device_mask = updated_clean_labels == largest_prop.label
        new_corners = _oriented_rect_corners(new_device_mask, cfg)

    final_corners = new_corners if flag else corners

    if return_debug:
        debug = {
            "median_thresholded": median_thresholded,
            "sobel_operated": sobel_operated,
            "binary": binary,
            "labels_to_dilate": labels_to_dilate,
            "post_dilation_mask": post_dilation_mask,
            "device_mask": device_mask,
            "organoid_region": organoid_region,
            "edges": edges,
            "reconstructed": reconstructed,
            "reconstructed_mask": reconstructed_mask,
            "flag": flag,
            "final_corners": final_corners,
            "new_corners": new_corners,
        }
        return plane, organoid_region, final_corners, debug

    return plane, organoid_region, final_corners


def detect_device_corners(in_focus_plane: np.ndarray, cfg: BatchConfig) -> np.ndarray:
    _, _, final_corners = segment_from_plane_conventional(
        in_focus_plane=in_focus_plane,
        cfg=cfg,
        mask_central_region=bool(cfg.mask_central_region),
        return_debug=False,
    )
    if final_corners is None:
        raise RuntimeError("Could not infer corners")
    return final_corners


def crop_rectified_stack_from_corners(stack: np.ndarray, corners_xy: np.ndarray):
    c = _order_corners_clockwise(corners_xy)
    w1 = np.hypot(c[1, 0] - c[0, 0], c[1, 1] - c[0, 1])
    w2 = np.hypot(c[2, 0] - c[3, 0], c[2, 1] - c[3, 1])
    h1 = np.hypot(c[3, 0] - c[0, 0], c[3, 1] - c[0, 1])
    h2 = np.hypot(c[2, 0] - c[1, 0], c[2, 1] - c[1, 1])
    width = int(max(w1, w2))
    height = int(max(h1, h2))
    if width <= 1 or height <= 1:
        raise RuntimeError("Invalid crop dimensions")
    dst = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=float)
    tform = ProjectiveTransform()
    if not tform.estimate(dst, c):
        raise RuntimeError("Could not estimate projective transform")
    return np.stack([warp(stack[z], tform, output_shape=(height, width), order=1, preserve_range=True) for z in range(stack.shape[0])], axis=0).astype(stack.dtype)


def _draw_polygon_on_rgb(rgb: np.ndarray, corners_xy: np.ndarray, color_rgb: tuple[int, int, int], downsample: int):
    if corners_xy is None:
        return
    c = np.asarray(corners_xy, dtype=float)
    if c.shape[0] < 4:
        return
    c = _order_corners_clockwise(c)
    c = c / float(max(1, int(downsample)))
    c = np.rint(c).astype(int)
    h, w = rgb.shape[:2]
    c[:, 0] = np.clip(c[:, 0], 0, max(0, w - 1))
    c[:, 1] = np.clip(c[:, 1], 0, max(0, h - 1))
    closed = np.vstack([c, c[0]])
    for i in range(closed.shape[0] - 1):
        x0, y0 = int(closed[i, 0]), int(closed[i, 1])
        x1, y1 = int(closed[i + 1, 0]), int(closed[i + 1, 1])
        rr, cc = line(y0, x0, y1, x1)
        rgb[rr, cc, 0] = int(color_rgb[0])
        rgb[rr, cc, 1] = int(color_rgb[1])
        rgb[rr, cc, 2] = int(color_rgb[2])


def make_geometry_overlay_image(in_focus_plane: np.ndarray, corners_xy: np.ndarray, outer_xy: np.ndarray, max_dim: int = 1024):
    base = np.asarray(to_gray(in_focus_plane), dtype=np.float32)
    h, w = base.shape[:2]
    max_dim = int(max(1, max_dim))
    downsample = int(max(1, math.ceil(max(h, w) / float(max_dim))))
    disp = base[::downsample, ::downsample]
    dmin = float(np.nanmin(disp))
    dmax = float(np.nanmax(disp))
    if not np.isfinite(dmin) or not np.isfinite(dmax) or dmax <= dmin:
        disp = np.zeros_like(disp, dtype=np.float32)
    else:
        disp = np.clip((disp - dmin) / (dmax - dmin), 0.0, 1.0)
    rgb = np.stack([(disp * 255).astype(np.uint8)] * 3, axis=-1)
    _draw_polygon_on_rgb(rgb, corners_xy, (255, 0, 0), downsample)
    _draw_polygon_on_rgb(rgb, outer_xy, (255, 255, 0), downsample)
    return rgb, downsample


def expand_rectangle_corners(corners_xy: np.ndarray, expand_x_px: float, expand_y_px: float):
    c = _order_corners_clockwise(corners_xy)
    center = c.mean(axis=0)
    e0 = c[1] - c[0]
    e1 = c[3] - c[0]
    l0 = float(np.linalg.norm(e0))
    l1 = float(np.linalg.norm(e1))
    if l0 <= 0 or l1 <= 0:
        raise RuntimeError("Invalid rectangle edge lengths")
    u0 = e0 / l0
    u1 = e1 / l1
    h0 = l0 * 0.5
    h1 = l1 * 0.5
    return np.array([
        center - (h0 + expand_x_px) * u0 - (h1 + expand_y_px) * u1,
        center + (h0 + expand_x_px) * u0 - (h1 + expand_y_px) * u1,
        center + (h0 + expand_x_px) * u0 + (h1 + expand_y_px) * u1,
        center - (h0 + expand_x_px) * u0 + (h1 + expand_y_px) * u1,
    ], dtype=float)


def image_resizer(stack_input: np.ndarray, src_voxel_um, ref_voxel_um, order: int = 3, use_gpu: bool = True) -> np.ndarray:
    stack = np.asarray(stack_input, dtype=np.float32)
    src = np.array(src_voxel_um, dtype=float)
    ref = np.array(ref_voxel_um, dtype=float)
    old_shape = np.array(stack.shape[:3], dtype=float)
    new_shape = np.maximum(1, np.rint(old_shape * (src / ref))).astype(int)
    zoom_factors = tuple(float(n) / float(o) for n, o in zip(new_shape, old_shape))
    if all(np.isclose(zf, 1.0) for zf in zoom_factors):
        return stack.astype(np.float32, copy=False)
    if use_gpu and HAS_CUPY:
        out = cpx_ndimage.zoom(cp.asarray(stack), zoom=zoom_factors, order=order).get()
    else:
        out = ndi_zoom(stack, zoom=zoom_factors, order=order, prefilter=False)
    return out.astype(np.float32)


def pix2pix_translation(stack_input: np.ndarray, model_path: str, device: str = "cuda", n_iter: int = 1) -> np.ndarray:
    class LegacyGenerator(nn.Module):
        def __init__(self, dropout_p: float = 0.4):
            super().__init__()
            self.unet = UNet(
                spatial_dims=3,
                in_channels=1,
                out_channels=1,
                channels=(32, 64, 128, 256, 512),
                strides=(1, 2, 2, 2, 1),
                num_res_units=3,
                dropout=float(dropout_p),
            )

        def forward(self, x):
            return F.relu(self.unet(x))

    checkpoint = torch.load(model_path, map_location="cpu")
    state_dict = checkpoint.get("state_dict", checkpoint)
    gen_items = [(k, v) for k, v in state_dict.items() if k.startswith("generator.")]
    if gen_items:
        gen_sd = {k[len("generator."):]: v for k, v in gen_items}
    elif "generator" in state_dict and isinstance(state_dict["generator"], dict):
        gen_sd = state_dict["generator"]
    else:
        raise RuntimeError("Could not find generator weights with prefix 'generator.'.")
    gen_sd = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in gen_sd.items()}

    hp = checkpoint.get("hyper_parameters", {}) if isinstance(checkpoint, dict) else {}
    dropout_p = float(hp.get("generator_dropout_p", 0.4)) if isinstance(hp, dict) else 0.4
    model = LegacyGenerator(dropout_p=dropout_p)
    model.load_state_dict(gen_sd, strict=True)
    model = model.to(device).eval()

    stack = np.asarray(stack_input, dtype=np.float32)
    den = float(stack.max() - stack.min())
    stack = np.zeros_like(stack, dtype=np.float32) if den <= 0 else (stack - stack.min()) / den
    input_tensor = torch.tensor(stack[None, None, ...], device="cpu").float()

    inferer = SlidingWindowInferer(roi_size=(32, 512, 512), overlap=(0.75, 0.25, 0.25), sw_batch_size=1, sw_device=device, device="cpu", progress=False)
    with torch.no_grad(), torch.amp.autocast(device_type="cuda" if str(device).startswith("cuda") else "cpu"):
        preds = torch.stack([inferer(inputs=input_tensor, network=model).to("cpu") for _ in range(max(1, int(n_iter)))])
        preds = torch.clip(preds, 0, 1)
        preds = torch.mean(preds, dim=0)

    out = preds.squeeze().cpu().numpy().astype(np.float32)
    model.to("cpu")
    del input_tensor, preds
    if torch.cuda.is_available() and str(device).startswith("cuda"):
        torch.cuda.empty_cache()
    return out


def u_net_segmentation(stack_input: np.ndarray, model_path: str, device: str = "cuda"):
    def _adapt_input_conv(in_chans, conv_weight):
        conv_type = conv_weight.dtype
        conv_weight = conv_weight.float()
        o_ch, i_ch, k_h, k_w = conv_weight.shape
        if in_chans == 1:
            if i_ch > 3:
                conv_weight = conv_weight.reshape(o_ch, i_ch // 3, 3, k_h, k_w)
                conv_weight = conv_weight.sum(dim=2, keepdim=False)
            else:
                conv_weight = conv_weight.sum(dim=1, keepdim=True)
        elif in_chans != 3:
            repeat = int(np.ceil(in_chans / 3))
            conv_weight = conv_weight.repeat(1, repeat, 1, 1)[:, :in_chans, :, :]
            conv_weight *= (3 / float(in_chans))
        return conv_weight.to(conv_type)

    model = smp.Unet(encoder_name="mit_b5", classes=1, in_channels=3, encoder_weights=None)
    if hasattr(model, "encoder") and hasattr(model.encoder, "patch_embed1"):
        new_weights = _adapt_input_conv(in_chans=1, conv_weight=model.encoder.patch_embed1.proj.weight)
        model.encoder.patch_embed1.proj = torch.nn.Conv2d(in_channels=1, out_channels=64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
        with torch.no_grad():
            model.encoder.patch_embed1.proj.weight = torch.nn.parameter.Parameter(new_weights)

    checkpoint = torch.load(model_path, map_location="cpu")
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    elif "state_dict" in checkpoint:
        sd = checkpoint["state_dict"]
        model.load_state_dict({(k[6:] if k.startswith("model.") else k): v for k, v in sd.items()})
    else:
        model.load_state_dict(checkpoint)

    model = model.to(device).eval()
    tensor = torch.tensor(np.asarray(stack_input, dtype=np.float32)[None, None, ...], device="cpu").float()
    shp = tensor.shape

    axial = SliceInferer(roi_size=(1024, 1024), sw_batch_size=4, progress=False, mode="gaussian", overlap=0.5, device="cpu", sw_device=device)
    with torch.no_grad():
        pa = torch.sigmoid(axial(tensor, model)).squeeze().to("cpu").numpy()

    if shp[2] < 256:
        pd = 256 - shp[2]
        pre = pd // 2
        post = pd - pre
        tensor = F.pad(tensor, (0, 0, 0, 0, pre, post), mode="constant", value=-1)

    cor = SliceInferer(roi_size=(256, 256), sw_batch_size=32, spatial_dim=1, progress=False, mode="gaussian", overlap=0.5, device="cpu", sw_device=device)
    sag = SliceInferer(roi_size=(256, 256), sw_batch_size=32, spatial_dim=2, progress=False, mode="gaussian", overlap=0.5, device="cpu", sw_device=device)
    with torch.no_grad():
        pc = torch.sigmoid(cor(tensor, model)).squeeze().to("cpu").numpy()
        ps = torch.sigmoid(sag(tensor, model)).squeeze().to("cpu").numpy()

    if shp[2] < 256:
        pd = 256 - shp[2]
        pre = pd // 2
        pc = pc[pre : pre + shp[2], :, :]
        ps = ps[pre : pre + shp[2], :, :]

    prob_map = np.mean(np.array([pa, pc, ps]), axis=0)
    filt = cpx_ndimage.median_filter(cp.asarray(prob_map), size=7).get() if HAS_CUPY else median(prob_map, footprint=np.ones((7, 7), dtype=bool))
    seg_mask = apply_hysteresis_threshold(filt, 0.2, 0.5).astype(np.uint8)

    model.to("cpu")
    del tensor
    if torch.cuda.is_available() and str(device).startswith("cuda"):
        torch.cuda.empty_cache()
    return prob_map.astype(np.float32), seg_mask


def crop_xy_equal_margin(arr: np.ndarray, crop_y_px: int, crop_x_px: int):
    arr = np.asarray(arr)
    y0 = int(max(0, crop_y_px))
    x0 = int(max(0, crop_x_px))
    y1 = int(arr.shape[1] - y0)
    x1 = int(arr.shape[2] - x0)
    if y1 <= y0 or x1 <= x0:
        raise RuntimeError(f"XY trim too large for shape {arr.shape}: crop_y={crop_y_px}, crop_x={crop_x_px}")
    return arr[:, y0:y1, x0:x1]


def compute_focus_votes_for_region(stack: np.ndarray, n_sampling: int, patch: int, region_mask: Optional[np.ndarray] = None):
    Z, H, W = stack.shape
    patch = max(5, int(patch))
    grid = max(4, int(n_sampling))
    half = patch // 2
    counts = np.zeros(Z, dtype=np.int32)
    if H <= patch or W <= patch:
        z_best = int(np.argmax([np.std(sobel(stack[z])) for z in range(Z)]))
        counts[z_best] = 1
        return counts
    ys = np.linspace(half, H - half - 1, grid).astype(int)
    xs = np.linspace(half, W - half - 1, grid).astype(int)
    for y in ys:
        for x in xs:
            if region_mask is not None and not bool(region_mask[y, x]):
                continue
            y0 = max(0, y - half)
            y1 = min(H, y + half)
            x0 = max(0, x - half)
            x1 = min(W, x + half)
            f = np.array([focus_score(stack[z, y0:y1, x0:x1]) for z in range(Z)], dtype=np.float32)
            counts[int(np.argmax(f))] += 1
    return counts


def counts_to_best(counts):
    counts = np.asarray(counts)
    if counts.size == 0 or int(np.sum(counts)) == 0:
        return None, 0
    z = int(np.argmax(counts))
    return z, int(counts[z])


def remap_counts_to_scaled_z(counts: np.ndarray, final_nz: int):
    counts = np.asarray(counts, dtype=np.int32)
    in_nz = int(counts.shape[0])
    out = np.zeros(int(max(1, final_nz)), dtype=np.int32)
    if in_nz <= 1:
        out[0] = int(counts.sum())
        return out
    for z_idx, c in enumerate(counts):
        if c <= 0:
            continue
        z_out = int(np.rint(float(z_idx) * float(final_nz - 1) / float(in_nz - 1)))
        out[z_out] += int(c)
    return out


def format_vote_table(counts: np.ndarray) -> str:
    counts = np.asarray(counts, dtype=np.int32)
    if counts.size == 0:
        return "none"
    hits = np.flatnonzero(counts > 0)
    if hits.size == 0:
        return "none"
    return "; ".join([f"z={int(i)}: votes={int(counts[i])}" for i in hits])


def format_voted_slices(counts: np.ndarray) -> str:
    counts = np.asarray(counts, dtype=np.int32)
    if counts.size == 0:
        return "none"
    hits = np.flatnonzero(counts > 0)
    if hits.size == 0:
        return "none"
    return ",".join([str(int(i)) for i in hits])


def format_slice_range(nz: int) -> str:
    nz = int(max(0, int(nz)))
    if nz <= 0:
        return "none"
    return f"0..{nz - 1}"


def sanitize_id(item: InputImage) -> str:
    stem = item.source_path.stem
    if item.source_kind == "lif":
        return f"{stem}__lif_{int(item.image_index):04d}".replace(" ", "_")
    return stem.replace(" ", "_")


def run_single(item: InputImage, cfg: BatchConfig) -> dict:
    image_id = sanitize_id(item)
    row = {
        "image_id": image_id,
        "source_file": str(item.source_path),
        "source_kind": item.source_kind,
        "image_index": item.image_index,
        "image_name": item.image_name,
        "status": "skipped",
        "skip_reason": "",
    }

    voxel = read_voxel_size_um(item)
    row["voxel_z_um"], row["voxel_y_um"], row["voxel_x_um"] = voxel
    if not has_valid_voxel_um(voxel):
        row["skip_reason"] = "missing_or_invalid_voxel_metadata"
        return row

    z_um, y_um, x_um = (float(voxel[0]), float(voxel[1]), float(voxel[2]))
    try:
        raw_stack = load_stack(item)
    except Exception as exc:
        row["skip_reason"] = f"load_failed: {type(exc).__name__}: {exc}"
        return row

    row["original_nz"] = int(raw_stack.shape[0])
    row["original_z_slices"] = f"0..{max(0, int(raw_stack.shape[0]) - 1)}"

    out_dir = cfg.output_dir / image_id
    out_dir.mkdir(parents=True, exist_ok=True)
    translation_path = out_dir / "translation.tif"
    prob_path = out_dir / "probability_map.tif"
    seg_path = out_dir / "segmentation_mask.tif"
    original_path = out_dir / "original_aligned.tif"
    overlay_path = out_dir / "geometry_overlay_downsampled.tif"
    npz_path = out_dir / "outputs.npz"
    meta_path = out_dir / "metadata.json"

    try:
        in_focus, _, _, _ = curved_plane_refocus(raw_stack, grid=cfg.focus_n_sampling, patch=cfg.focus_patch)
        corners = detect_device_corners(in_focus, cfg)
        expand_x_px = float(cfg.device_width_um) / float(x_um)
        expand_y_px = float(cfg.device_width_um) / float(y_um)
        outer = expand_rectangle_corners(corners, expand_x_px, expand_y_px)
        cropped_stack = crop_rectified_stack_from_corners(raw_stack, outer)
    except Exception as exc:
        row["skip_reason"] = f"geometry_failed: {type(exc).__name__}: {exc}"
        return row

    overlay_rgb, overlay_downsample = make_geometry_overlay_image(
        in_focus_plane=in_focus,
        corners_xy=corners,
        outer_xy=outer,
        max_dim=int(cfg.overlay_max_dim),
    )
    tifffile.imwrite(str(overlay_path), overlay_rgb.astype(np.uint8))
    row["geometry_overlay_path"] = str(overlay_path)
    row["geometry_overlay_downsample_factor"] = int(overlay_downsample)

    geom_counts = compute_focus_votes_for_region(cropped_stack, cfg.focus_n_sampling, cfg.focus_patch)
    geom_z, geom_votes = counts_to_best(geom_counts)

    erode_y = int(np.rint(float(cfg.device_width_um) / float(y_um)))
    erode_x = int(np.rint(float(cfg.device_width_um) / float(x_um)))
    inner_mask = np.zeros(cropped_stack.shape[1:], dtype=bool)
    if (cropped_stack.shape[1] - 2 * erode_y) > 4 and (cropped_stack.shape[2] - 2 * erode_x) > 4:
        inner_mask[erode_y : cropped_stack.shape[1] - erode_y, erode_x : cropped_stack.shape[2] - erode_x] = True
    eroded_counts = compute_focus_votes_for_region(cropped_stack, cfg.focus_n_sampling, cfg.focus_patch, region_mask=inner_mask if np.any(inner_mask) else None)
    eroded_z, eroded_votes = counts_to_best(eroded_counts)

    row["geometry_focus_z"] = geom_z
    row["geometry_focus_votes"] = int(geom_votes)
    row["geometry_eroded_focus_z"] = eroded_z
    row["geometry_eroded_focus_votes"] = int(eroded_votes)
    row["geometry_focus_vote_table"] = format_vote_table(geom_counts)
    row["geometry_focus_voted_slices"] = format_voted_slices(geom_counts)
    row["geometry_eroded_focus_vote_table"] = format_vote_table(eroded_counts)
    row["geometry_eroded_focus_voted_slices"] = format_voted_slices(eroded_counts)
    row["geometry_stack_kept_slices_for_segmentation"] = format_slice_range(int(cropped_stack.shape[0]))

    try:
        pix_in = image_resizer(cropped_stack, (z_um, y_um, x_um), cfg.pix2pix_voxel_um, order=3, use_gpu=cfg.use_gpu_resize)
        translation = pix2pix_translation(pix_in, str(cfg.pix2pix_ckpt), device=cfg.device, n_iter=1)
        unet_in = image_resizer(translation, cfg.pix2pix_voxel_um, cfg.unet_input_voxel_um, order=3, use_gpu=cfg.use_gpu_resize)
        prob_map, seg_mask = u_net_segmentation(unet_in, str(cfg.seg_ckpt), device=cfg.device)
    except Exception as exc:
        row["skip_reason"] = f"model_inference_failed: {type(exc).__name__}: {exc}"
        return row

    translation_out = image_resizer(translation, cfg.pix2pix_voxel_um, cfg.output_voxel_um, order=1, use_gpu=cfg.use_gpu_resize).astype(np.float32)
    prob_out = image_resizer(prob_map, cfg.unet_input_voxel_um, cfg.output_voxel_um, order=1, use_gpu=cfg.use_gpu_resize).astype(np.float32)
    seg_out = image_resizer(seg_mask.astype(np.float32), cfg.unet_input_voxel_um, cfg.output_voxel_um, order=0, use_gpu=cfg.use_gpu_resize).astype(np.uint8)
    original_out = image_resizer(cropped_stack, (z_um, y_um, x_um), cfg.output_voxel_um, order=1, use_gpu=cfg.use_gpu_resize).astype(np.float32)

    target_shape = tuple(int(v) for v in seg_out.shape[:3])
    if tuple(translation_out.shape[:3]) != target_shape:
        translation_out = ndi_zoom(translation_out, zoom=(target_shape[0] / translation_out.shape[0], target_shape[1] / translation_out.shape[1], target_shape[2] / translation_out.shape[2]), order=1, prefilter=False).astype(np.float32)
    if tuple(prob_out.shape[:3]) != target_shape:
        prob_out = ndi_zoom(prob_out, zoom=(target_shape[0] / prob_out.shape[0], target_shape[1] / prob_out.shape[1], target_shape[2] / prob_out.shape[2]), order=1, prefilter=False).astype(np.float32)
    if tuple(original_out.shape[:3]) != target_shape:
        original_out = ndi_zoom(original_out, zoom=(target_shape[0] / original_out.shape[0], target_shape[1] / original_out.shape[1], target_shape[2] / original_out.shape[2]), order=1, prefilter=False).astype(np.float32)

    crop_y = int(np.rint(float(cfg.device_width_um) / float(cfg.output_voxel_um[1])))
    crop_x = int(np.rint(float(cfg.device_width_um) / float(cfg.output_voxel_um[2])))
    try:
        translation_crop = crop_xy_equal_margin(translation_out, crop_y, crop_x).astype(np.float32)
        prob_crop = crop_xy_equal_margin(prob_out, crop_y, crop_x).astype(np.float32)
        seg_crop = crop_xy_equal_margin(seg_out, crop_y, crop_x).astype(np.uint8)
        original_crop = crop_xy_equal_margin(original_out, crop_y, crop_x).astype(np.float32)
    except Exception as exc:
        row["skip_reason"] = f"xy_crop_failed: {type(exc).__name__}: {exc}"
        return row

    final_nz = int(seg_crop.shape[0])
    geom_counts_final = remap_counts_to_scaled_z(geom_counts, final_nz)
    eroded_counts_final = remap_counts_to_scaled_z(eroded_counts, final_nz)
    final_geom_z, final_geom_votes = counts_to_best(geom_counts_final)
    final_eroded_z, final_eroded_votes = counts_to_best(eroded_counts_final)

    row["final_nz"] = final_nz
    row["output_voxel_um"] = str(tuple(float(v) for v in cfg.output_voxel_um))
    row["final_geometry_focus_z"] = final_geom_z
    row["final_geometry_focus_votes"] = int(final_geom_votes)
    row["final_geometry_eroded_focus_z"] = final_eroded_z
    row["final_geometry_eroded_focus_votes"] = int(final_eroded_votes)
    row["final_geometry_focus_vote_table"] = format_vote_table(geom_counts_final)
    row["final_geometry_focus_voted_slices"] = format_voted_slices(geom_counts_final)
    row["final_geometry_eroded_focus_vote_table"] = format_vote_table(eroded_counts_final)
    row["final_geometry_eroded_focus_voted_slices"] = format_voted_slices(eroded_counts_final)
    row["segmentation_input_kept_slices"] = format_slice_range(int(unet_in.shape[0]))
    row["segmentation_output_kept_slices"] = format_slice_range(final_nz)

    tifffile.imwrite(str(translation_path), translation_crop.astype(np.float32))
    tifffile.imwrite(str(prob_path), prob_crop.astype(np.float32))
    tifffile.imwrite(str(seg_path), seg_crop.astype(np.uint8))
    tifffile.imwrite(str(original_path), original_crop.astype(np.float32))
    np.savez_compressed(
        str(npz_path),
        original_aligned=original_crop.astype(np.float32),
        translation=translation_crop.astype(np.float32),
        probability_map=prob_crop.astype(np.float32),
        segmentation_mask=seg_crop.astype(np.uint8),
        voxel_um=np.asarray(cfg.output_voxel_um, dtype=np.float32),
    )

    metadata = {
        "image_id": image_id,
        "source_file": str(item.source_path),
        "source_kind": item.source_kind,
        "image_index": item.image_index,
        "image_name": item.image_name,
        "voxel_um_source": [z_um, y_um, x_um],
        "device_width_um": float(cfg.device_width_um),
        "output_voxel_um": [float(v) for v in cfg.output_voxel_um],
        "geometry_focus_z": geom_z,
        "geometry_focus_votes": int(geom_votes),
        "geometry_eroded_focus_z": eroded_z,
        "geometry_eroded_focus_votes": int(eroded_votes),
        "geometry_focus_vote_table": format_vote_table(geom_counts),
        "geometry_focus_voted_slices": format_voted_slices(geom_counts),
        "geometry_eroded_focus_vote_table": format_vote_table(eroded_counts),
        "geometry_eroded_focus_voted_slices": format_voted_slices(eroded_counts),
        "geometry_stack_kept_slices_for_segmentation": format_slice_range(int(cropped_stack.shape[0])),
        "final_geometry_focus_z": final_geom_z,
        "final_geometry_focus_votes": int(final_geom_votes),
        "final_geometry_eroded_focus_z": final_eroded_z,
        "final_geometry_eroded_focus_votes": int(final_eroded_votes),
        "final_geometry_focus_vote_table": format_vote_table(geom_counts_final),
        "final_geometry_focus_voted_slices": format_voted_slices(geom_counts_final),
        "final_geometry_eroded_focus_vote_table": format_vote_table(eroded_counts_final),
        "final_geometry_eroded_focus_voted_slices": format_voted_slices(eroded_counts_final),
        "segmentation_input_kept_slices": format_slice_range(int(unet_in.shape[0])),
        "segmentation_output_kept_slices": format_slice_range(final_nz),
        "original_aligned_tif": str(original_path),
        "geometry_overlay_downsampled_tif": str(overlay_path),
        "geometry_overlay_downsample_factor": int(overlay_downsample),
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    row["translation_path"] = str(translation_path)
    row["probability_map_path"] = str(prob_path)
    row["segmentation_mask_path"] = str(seg_path)
    row["original_aligned_path"] = str(original_path)
    row["npz_path"] = str(npz_path)
    row["metadata_path"] = str(meta_path)
    row["status"] = "ok"
    row["skip_reason"] = ""
    return row


def run_batch(config: BatchConfig) -> pd.DataFrame:
    config.output_dir.mkdir(parents=True, exist_ok=True)
    items = iterate_input_images(config.input_dir)
    rows = []
    for i, item in enumerate(items, start=1):
        print(f"[{i}/{len(items)}] Processing {item.source_path} :: {item.image_name}")
        row = run_single(item, config)
        rows.append(row)
        if row.get("status") != "ok":
            print(f"  -> skipped: {row.get('skip_reason', '')}")
        else:
            print("  -> done")

    df = pd.DataFrame(rows)
    csv_path = config.output_dir / "run_summary.csv"
    df.to_csv(csv_path, index=False)

    run_config = {
        "input_dir": str(config.input_dir),
        "output_dir": str(config.output_dir),
        "pix2pix_ckpt": str(config.pix2pix_ckpt),
        "seg_ckpt": str(config.seg_ckpt),
        "device": str(config.device),
        "device_width_um": float(config.device_width_um),
        "output_voxel_um": [float(v) for v in config.output_voxel_um],
        "pix2pix_voxel_um": [float(v) for v in config.pix2pix_voxel_um],
        "unet_input_voxel_um": [float(v) for v in config.unet_input_voxel_um],
        "mask_central_region": bool(config.mask_central_region),
        "overlay_max_dim": int(config.overlay_max_dim),
        "n_images_found": int(len(items)),
        "n_ok": int((df["status"] == "ok").sum()) if "status" in df.columns else 0,
        "n_skipped": int((df["status"] != "ok").sum()) if "status" in df.columns else 0,
    }
    with open(config.output_dir / "run_config.json", "w", encoding="utf-8") as f:
        json.dump(run_config, f, indent=2)

    print(f"Saved summary: {csv_path}")
    return df


def load_saved_run(output_dir: Path):
    output_dir = Path(output_dir)
    csv_path = output_dir / "run_summary.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing summary CSV: {csv_path}")
    df = pd.read_csv(csv_path)
    loaded = {}
    for _, row in df.iterrows():
        if str(row.get("status", "")) != "ok":
            continue
        image_id = str(row["image_id"])
        translation_path = Path(row["translation_path"])
        prob_path = Path(row["probability_map_path"])
        seg_path = Path(row["segmentation_mask_path"])
        original_path = Path(row["original_aligned_path"]) if "original_aligned_path" in row and isinstance(row["original_aligned_path"], str) else None
        if translation_path.exists() and prob_path.exists() and seg_path.exists():
            loaded[image_id] = {
                "translation": tifffile.imread(str(translation_path)),
                "probability_map": tifffile.imread(str(prob_path)),
                "segmentation_mask": tifffile.imread(str(seg_path)),
            }
            if original_path is not None and original_path.exists():
                loaded[image_id]["original_aligned"] = tifffile.imread(str(original_path))
    return df, loaded


print("Pipeline code loaded in notebook. You can now run the config cell.")

Pipeline code loaded in notebook. You can now run the config cell.


In [17]:
# Set your test folders here
input_dir = Path(r"Z:\Bel\Vascumap_Example_Lifs\training_data\original_images\test_in")
output_dir = Path(r"Z:\Bel\Vascumap_Example_Lifs\training_data\original_images\test_out")

pix2pix_ckpt = Path(r"luca_models\epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt")
seg_ckpt = Path(r"luca_models\best_full.pth")

device_width_um = 40.0
output_voxel_um = (2.0, 2.0, 2.0)

config = BatchConfig(
    input_dir=input_dir,
    output_dir=output_dir,
    pix2pix_ckpt=pix2pix_ckpt,
    seg_ckpt=seg_ckpt,
    device="cuda",
    device_width_um=device_width_um,
    output_voxel_um=output_voxel_um,
    pix2pix_voxel_um=(5.0, 2.0, 2.0),
    unet_input_voxel_um=(2.0, 2.0, 2.0),
    focus_n_sampling=10,
    focus_patch=50,
    use_gpu_resize=True,
)

print("Input dir:", config.input_dir)
print("Output dir:", config.output_dir)
print("Device width (um):", config.device_width_um)
print("Output voxel (z,y,x) um:", config.output_voxel_um)

Input dir: Z:\Bel\Vascumap_Example_Lifs\training_data\original_images\test_in
Output dir: Z:\Bel\Vascumap_Example_Lifs\training_data\original_images\test_out
Device width (um): 40.0
Output voxel (z,y,x) um: (2.0, 2.0, 2.0)


In [ ]:
# Run full batch (overnight-safe)
df = run_batch(config)
print("Rows:", len(df))
display(df.head(20))

[1/1] Processing Z:\Bel\Vascumap_Example_Lifs\training_data\original_images\test_in\20250619_Vascumap_Dev25_11_Daisy10.tif :: 20250619_Vascumap_Dev25_11_Daisy10.tif


C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_38060\737077394.py:513: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `ProjectiveTransform.from_estimate` class constructor instead.
  if not tform.estimate(dst, c):


In [4]:
df

,image_id,source_file,source_kind,image_index,image_name,status,skip_reason,voxel_z_um,voxel_y_um,voxel_x_um,...,output_voxel_um,final_geometry_focus_z,final_geometry_focus_votes,final_geometry_eroded_focus_z,final_geometry_eroded_focus_votes,translation_path,probability_map_path,segmentation_mask_path,npz_path,metadata_path
0,20250619_Vascumap_Dev25_11_Daisy10,Z:\Bel\Vascumap_Example_Lifs\training_data\ori...,tif,None,20250619_Vascumap_Dev25_11_Daisy10.tif,ok,,7.99811,1.300001,1.300001,...,"(2.0, 2.0, 2.0)",32,10,32,8,Z:\Bel\Vascumap_Example_Lifs\training_data\ori...,Z:\Bel\Vascumap_Example_Lifs\training_data\ori...,Z:\Bel\Vascumap_Example_Lifs\training_data\ori...,Z:\Bel\Vascumap_Example_Lifs\training_data\ori...,Z:\Bel\Vascumap_Example_Lifs\training_data\ori...


In [ ]:
# Reload saved outputs tomorrow
summary_df, loaded = load_saved_run(output_dir)
print("Summary rows:", len(summary_df))
print("Loaded image volumes:", len(loaded))
display(summary_df.head(20))

In [ ]:
# Optional: inspect one loaded item
if loaded:
    one_id = next(iter(loaded.keys()))
    data = loaded[one_id]
    print("Image ID:", one_id)
    print("translation:", data["translation"].shape, data["translation"].dtype)
    print("probability_map:", data["probability_map"].shape, data["probability_map"].dtype)
    print("segmentation_mask:", data["segmentation_mask"].shape, data["segmentation_mask"].dtype)
else:
    print("No successful outputs loaded.")